# SurahChain 114 — تدريب على Kaggle (ليلي)

## قبل التشغيل (مرة واحدة)
1. **Settings → Accelerator → GPU T4** (أو Dual T4 إن وُجد)
2. **Settings → Internet → ON**
3. **Add-ons → Secrets** → أنشئ سرّاً:
   - الاسم: `GITHUB_TOKEN`
   - القيمة: Personal Access Token من GitHub (صلاحية `repo`)
4. عدّل خلية **الإعدادات** إن لزم
5. من القائمة: **Save Version → Save & Run All**
6. أغلق المتصفح — التدريب يكمل في الخلفية

## ماذا يفعل؟
- يسحب المستودع، يجهّز بيانات عربية، يدرّب **medium** (`d_model=256`)
- **سلسلة 114 كما هي** (لا دمج / لا تغيير أبعاد)
- يحفظ checkpoint كل عصر ويرفع إلى GitHub

## الاستكمال لاحقاً
ضع `SCN_FRESH = False` ثم **Save & Run All** مرة أخرى.


In [ ]:
# =========================
# إعدادات — d_model=128 (استكمال من آخر تدريب)
# =========================
SCN_PRESET = "small"    # d_model=128 — لا تبدأ من الصفر
SCN_N = 30000            # نفس بيانات آخر جولة (أو 60000 إن وسّعت الكاش)
SCN_EPOCHS = 30          # حقب إضافية
SCN_BATCH = 24
SCN_FRESH = False        # استكمال من checkpoint الموجود
SCN_COMPILE = True
SCN_QK_NORM = True       # إن سبب عدم تطابق: False
SCN_GATED_ATTN = True    # إن سبب عدم تطابق: False
AUTO_PUSH = True

REPO = "aliahmed369000000-ai/Neural-Service-Mesh"
BRANCH = "main"

print("=" * 50)
print("preset:", SCN_PRESET, "→ d_model=128")
print("N:", SCN_N, "| epochs:", SCN_EPOCHS, "| batch:", SCN_BATCH)
print("FRESH:", SCN_FRESH, "(False = استكمال)")
print("سلسلة 114: كما هي")
print("=" * 50)


In [ ]:
import os, sys, torch
print("Python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU{i}:", torch.cuda.get_device_name(i))
else:
    print("تحذير: لا GPU — فعّل Accelerator من Settings")
!pip -q install -U datasets huggingface_hub


In [ ]:
import os, pathlib, subprocess

GITHUB_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
    print("✓ التوكن من Kaggle Secrets")
except Exception as e:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
    print("Secrets:", e)

if not GITHUB_TOKEN:
    raise SystemExit("أضف Secret باسم GITHUB_TOKEN من Add-ons → Secrets")

work = pathlib.Path("/kaggle/working/Neural-Service-Mesh")
url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"

if work.exists():
    os.chdir(work)
    subprocess.run(["git", "remote", "set-url", "origin", url], check=False)
    subprocess.run(["git", "pull", "origin", BRANCH], check=False)
    print("✓ تحديث المستودع")
else:
    os.chdir("/kaggle/working")
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "Neural-Service-Mesh"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr)
        raise SystemExit("فشل clone — تحقق من التوكن")
    os.chdir(work)
    print("✓ استنساخ المستودع")

subprocess.run(["git", "config", "user.email", "nsm-bot@users.noreply.github.com"], check=False)
subprocess.run(["git", "config", "user.name", "NSM Bot"], check=False)
print("cwd:", os.getcwd())


In [ ]:
import os
os.environ["SCN_N"] = str(SCN_N)
print("تحضير بيانات Pre-train... N=", SCN_N)
!python experiments/surah_chain_network/prepare_pretrain_data.py


In [ ]:
import os
os.environ["SCN_PRESET"] = SCN_PRESET
os.environ["SCN_N"] = str(SCN_N)
os.environ["SCN_EPOCHS"] = str(SCN_EPOCHS)
os.environ["SCN_BATCH"] = str(SCN_BATCH)
os.environ["SCN_FRESH"] = "1" if SCN_FRESH else "0"
os.environ["SCN_COMPILE"] = "1" if SCN_COMPILE else "0"
os.environ["SCN_QK_NORM"] = "1" if SCN_QK_NORM else "0"
os.environ["SCN_GATED_ATTN"] = "1" if SCN_GATED_ATTN else "0"
os.environ["SCN_CHAIN_SCALE"] = "1"

print("بدء التدريب...", SCN_PRESET, "fresh=", SCN_FRESH)
!python experiments/surah_chain_network/train_pretrain_torch.py
print("انتهى التدريب")


In [ ]:
from pathlib import Path
import subprocess, json

exp = Path("experiments/surah_chain_network")
ckpt = exp / "checkpoints"
print("--- الملفات ---")
for p in sorted(ckpt.glob("*")):
    if p.is_file():
        print(f"  {p.name}: {p.stat().st_size/1e6:.2f} MB")

for state in list(ckpt.glob("pretrain_state_*.json")) + list(ckpt.glob("pretrain_torch_state.json")):
    try:
        d = json.loads(state.read_text())
        print(f"\n[{state.name}]")
        for k in ("best_loss", "epochs_completed", "d_model", "chain_scale", "n_sentences", "global_step", "tag"):
            if k in d:
                print(f"  {k}: {d[k]}")
    except Exception as e:
        print("state:", e)

if not AUTO_PUSH:
    print("تخطي الرفع")
else:
    files = list(ckpt.glob("*.pt")) + list(ckpt.glob("pretrain_state_*.json")) + list(ckpt.glob("pretrain_torch_state.json")) + list(exp.glob("tokenizer_vocab_pretrain*.json"))
    for f in files:
        if f.exists() and f.stat().st_size > 50:
            subprocess.run(["git", "add", "-f", str(f)], check=False)
            print("add", f.name)
    st = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)
    if not st.stdout.strip():
        print("لا تغييرات")
    else:
        msg = f"Kaggle: SurahChain {SCN_PRESET} pretrain (N={SCN_N}, epochs={SCN_EPOCHS})"
        subprocess.run(["git", "commit", "-m", msg], check=False)
        r = subprocess.run(["git", "push", "origin", BRANCH], capture_output=True, text=True)
        print(r.stdout or "")
        print(r.stderr or "")
        print("✓ تم الرفع" if r.returncode == 0 else "✗ فشل الرفع")


## بعد الاستيقاظ
1. تأكد أن Version حالتها **Success**
2. على GitHub راجع `best_loss` في `pretrain_state_*.json`
3. للاستكمال: `SCN_FRESH = False` ثم Save & Run All

هذه الجولة تشغّل بنية **114** وتقيس الـloss (هدف بحثي).
